# 02 | Portfolio conflicts and next occasions

**Author: Chanakya**

Combine dated F1 and football evidence with the ATP slot sample. MotoGP offsets remain quarantined where a circuit-specific clock interpretation has not been validated.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='02_portfolio_clashes'
shared.ACTIVE_SOURCES=['v3_jolpica_races_2026', 'v4_laliga_2526', 'v4_laliga_2627', 'v4_motogp_finished_events', 'v4_motogp_future_events', 'v4_motogp_report_ara_rac', 'v4_motogp_report_ara_spr', 'v4_motogp_report_bra_rac', 'v4_motogp_report_bra_spr', 'v4_motogp_report_cat_rac', 'v4_motogp_report_cat_rac2', 'v4_motogp_report_cat_spr', 'v4_motogp_report_cze_rac', 'v4_motogp_report_cze_spr', 'v4_motogp_report_fra_rac', 'v4_motogp_report_fra_spr', 'v4_motogp_report_gbr_rac', 'v4_motogp_report_gbr_spr', 'v4_motogp_report_ger_rac', 'v4_motogp_report_ger_spr', 'v4_motogp_report_hun_rac', 'v4_motogp_report_hun_spr', 'v4_motogp_report_ita_rac', 'v4_motogp_report_ita_spr', 'v4_motogp_report_ned_rac', 'v4_motogp_report_ned_spr', 'v4_motogp_report_rsm_rac', 'v4_motogp_report_rsm_spr', 'v4_motogp_report_spa_rac', 'v4_motogp_report_spa_spr', 'v4_motogp_report_tha_rac', 'v4_motogp_report_tha_spr', 'v4_motogp_report_usa_rac', 'v4_motogp_report_usa_spr', 'v4_motogp_sessions_ara', 'v4_motogp_sessions_aus', 'v4_motogp_sessions_aut', 'v4_motogp_sessions_bra', 'v4_motogp_sessions_cat', 'v4_motogp_sessions_code', 'v4_motogp_sessions_cze', 'v4_motogp_sessions_fra', 'v4_motogp_sessions_gbr', 'v4_motogp_sessions_ger', 'v4_motogp_sessions_hun', 'v4_motogp_sessions_ina', 'v4_motogp_sessions_ita', 'v4_motogp_sessions_jpn', 'v4_motogp_sessions_mal', 'v4_motogp_sessions_ned', 'v4_motogp_sessions_por', 'v4_motogp_sessions_qat', 'v4_motogp_sessions_rsm', 'v4_motogp_sessions_spa', 'v4_motogp_sessions_tha', 'v4_motogp_sessions_usa', 'v4_motogp_sessions_val']

Offline inputs: raw-v5-2026-09-14 | Author: Chanakya


## 1. F1 and football clocks
F1 feed fields are scheduled UTC. Football-Data clocks use a Europe/London interpretation supported by winter and summer official fixture checks. Retain that interpretation as a documented inference.

In [2]:
f1=pd.DataFrame([dict(event=r['raceName'],round=r['round'],start_utc=pd.Timestamp(r['date']+'T'+r['time']),source_id='v3_jolpica_races_2026',sport='F1') for r in rawjson('v3_jolpica_races_2026')['MRData']['RaceTable']['Races']])
f1['start_ist']=pd.to_datetime(f1.start_utc,utc=True).dt.tz_convert('Asia/Kolkata');table(f1,'f1_races',True)
football=[]
for sid in ['v4_laliga_2526','v4_laliga_2627']:
 d=pd.read_csv(path(sid));d['start_utc']=pd.to_datetime(d.Date+' '+d.Time,format='%d/%m/%Y %H:%M').dt.tz_localize('Europe/London',ambiguous='raise',nonexistent='raise').dt.tz_convert('UTC');d['source_id']=sid;football.append(d)
football=pd.concat(football,ignore_index=True);football=football[(football.start_utc>=pd.Timestamp('2026-01-01',tz='UTC'))&(football.start_utc<pd.Timestamp('2026-09-13',tz='UTC'))].copy()
football['event']=football.HomeTeam+' vs '+football.AwayTeam;football['sport']='La Liga';football['start_ist']=football.start_utc.dt.tz_convert('Asia/Kolkata')
football=football[['event','HomeTeam','AwayTeam','start_utc','start_ist','sport','source_id']];table(football,'football_fixtures',True)
display(pd.DataFrame({'sport':['F1','La Liga'],'rows':[len(f1),len(football)],'coverage':['Current 2026 schedule','2026 observations through 7 September']}))

,sport,rows,coverage
0,F1,23,Current 2026 schedule
1,La Liga,250,2026 observations through 7 September


## 2. MotoGP: expose the timezone defect before using the data
All 177 session records are retained. The Assen race is independently checked against its official UTC+2 timetable. Other session offsets are not silently corrected. All 29 report excerpts are exported as local-clock evidence with restarts preserved.

In [3]:
moto=[]
for sid,src in SOURCES.items():
 if sid.startswith('v4_motogp_sessions_') and src.get('extension')=='json':
  for r in rawjson(sid):moto.append(dict(session_id=r['id'],event=r['event']['name'],event_code=r['event']['short_name'],type=r['type'],date_literal=r['date'],status=r['status'],source_id=sid,utc_usable=False))
moto=pd.DataFrame(moto);table(moto,'motogp_sessions_quarantined',True)
qa=json.loads((ROOT/'data/manifests/recovery_v4_qa.json').read_text());logs=[]
for r in qa['motogp']['official_reports']:
 for p in r['race_start_evidence']:
  for clock in p['race_start_clocks']:logs.append(dict(source_id=r['source_id'],pdf_page=p['page'],local_clock=clock,semantics='Race-control clock, possible repeated restart history'))
table(pd.DataFrame(logs),'02_motogp_actual_clock_evidence')
assen=moto[(moto.event_code=='NED')&(moto.type=='RAC')].iloc[0]
literal=pd.Timestamp(assen.date_literal);correct=pd.Timestamp(literal.tz_localize(None),tz='Europe/Amsterdam').tz_convert('UTC')
comparison=pd.DataFrame({'interpretation':['Literal API offset','Official local-time interpretation'],'IST':[literal.tz_convert('Asia/Kolkata'),correct.tz_convert('Asia/Kolkata')]});display(table(comparison,'02_assen_timezone_audit'))
display(table(moto.groupby(['type','status']).size().rename('sessions').reset_index(),'02_motogp_coverage'))

,interpretation,IST
0,Literal API offset,2026-06-28 19:30:00+05:30
1,Official local-time interpretation,2026-06-28 17:30:00+05:30


,type,status,sessions
0,FP,FINISHED,28
1,FP,NOT-STARTED,16
2,PR,FINISHED,14
3,PR,NOT-STARTED,8
4,Q,FINISHED,28
5,Q,NOT-STARTED,16
6,RAC,FINISHED,15
7,RAC,NOT-STARTED,8
8,SPR,FINISHED,14
9,SPR,NOT-STARTED,8


## 3. ATP/F1/football overlap sensitivity
No actual ATP durations are available. Compute overlap under 90/150/210-minute ATP and alternative rival duration scenarios. A clash exists only for the selected fixture basket, not all fans of that sport. Weekend co-occurrence is a separate, weaker measure.

In [4]:
slots=read('atp_slots');slots['start_utc']=pd.to_datetime(slots.start_utc,utc=True)
rivals=pd.concat([f1[['event','sport','start_utc','source_id']],football[['event','sport','start_utc','source_id']]],ignore_index=True)
clashes=[]
for a in slots[slots['round']=='Singles final'].itertuples():
 nearby=rivals[(rivals.start_utc-a.start_utc).abs()<=pd.Timedelta(hours=8)]
 for b in nearby.itertuples():
  for da in CFG['atp_duration_scenarios_minutes']:
   for db in (CFG['f1_duration_scenarios_minutes'] if b.sport=='F1' else CFG['football_duration_scenarios_minutes']):
    overlap=max(0,(min(a.start_utc+pd.Timedelta(minutes=da),b.start_utc+pd.Timedelta(minutes=db))-max(a.start_utc,b.start_utc)).total_seconds()/60)
    clashes.append(dict(atp_event=a.event,rival=b.event,sport=b.sport,atp_duration=da,rival_duration=db,overlap_minutes=overlap,atp_source=a.source_id,rival_source=b.source_id))
clashes=pd.DataFrame(clashes);table(clashes,'02_clash_scenarios')
cs=clashes.groupby(['atp_event','rival','sport']).overlap_minutes.agg(['min','max']).reset_index();display(table(cs,'02_clash_bounds'))
base=clashes[(clashes.atp_duration==150)&(((clashes.sport=='F1')&(clashes.rival_duration==120))|((clashes.sport=='La Liga')&(clashes.rival_duration==105)))]
plt.figure(figsize=(10,5));g=base[base.overlap_minutes>0].sort_values('overlap_minutes').tail(12);plt.barh(g.atp_event+' / '+g.rival,g.overlap_minutes);plt.xlabel('Scenario overlap minutes');plt.title('Selected finals can compete with specific football fixtures');fig('02_selected_clashes','Assumed durations, scheduled starts. Partial final and football sample, no actual viewing or audience overlap.')
# Event-date co-occurrence does not imply hourly overlap.
ev=rawjson('v4_motogp_finished_events')+rawjson('v4_motogp_future_events');ev={r['id']:r for r in ev if not r['test']}
co=[]
for a in slots[slots['round']=='Singles final'].itertuples():
 local=pd.Timestamp(a.date_local).date()
 for e in ev.values():
  if pd.Timestamp(e['date_start']).date()<=local<=pd.Timestamp(e['date_end']).date():co.append(dict(atp_event=a.event,motogp_event=e['name'],type='Event-date co-occurrence only',atp_source=a.source_id))
display(table(pd.DataFrame(co),'02_motogp_weekend_cooccurrence'))
check('02_portfolio',{'f1_23_rounds':len(f1)==23,'football_250_rows':len(football)==250,'motogp_177_unique':len(moto)==177 and moto.session_id.is_unique,'assen_offset_difference_2h':(literal-correct).total_seconds()==7200,'no_negative_overlap':bool((clashes.overlap_minutes>=0).all())})

,atp_event,rival,sport,min,max
0,Brisbane,Levante vs Espanol,La Liga,0.000,0.000
1,Brisbane,Vallecano vs Mallorca,La Liga,0.000,0.000
2,Doha,Ath Madrid vs Espanol,La Liga,0.000,90.000
3,Doha,Betis vs Vallecano,La Liga,0.000,0.000
4,Doha,Osasuna vs Real Madrid,La Liga,75.000,90.000
...,...,...,...,...,...
37,Rotterdam,Oviedo vs Ath Bilbao,La Liga,15.000,30.000
38,Rotterdam,Vallecano vs Ath Madrid,La Liga,45.000,120.000
39,Winston-Salem,Levante vs Betis,La Liga,0.000,0.000
40,Winston-Salem,Sevilla vs Ath Madrid,La Liga,75.000,90.000


<Figure size 1000x500 with 1 Axes>

,atp_event,motogp_event,type,atp_source
0,Dubai,GRAND PRIX OF THAILAND,Event-date co-occurrence only,dubai_2026_official_op
1,Miami,GRAND PRIX OF THE UNITED STATES,Event-date co-occurrence only,miami_2026_official_op
2,Rome,GRAND PRIX OF CATALONIA,Event-date co-occurrence only,rome_2026_official_op
3,Halle,GRAND PRIX OF CZECHIA,Event-date co-occurrence only,halle_2026_official_op
4,Winston-Salem,GRAND PRIX OF ARAGON,Event-date co-occurrence only,winston_oop_pdf


,check,passed
0,f1_23_rounds,True
1,football_250_rows,True
2,motogp_177_unique,True
3,assen_offset_difference_2h,True
4,no_negative_overlap,True


## Decision passed forward
Use sport-specific conflict suppression with the customer’s chosen race or club. A shared weekend may offer a bundle opportunity, but hourly overlap can make the bundle less usable. Broad MotoGP hourly clash claims remain outside the verified evidence. F1/football overlap tables are sensitivity results, not realized collisions.

## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [5]:
references=source_table(['v3_jolpica_races_2026', 'v4_laliga_2526', 'v4_laliga_2627', 'v4_motogp_finished_events', 'v4_motogp_future_events', 'v4_motogp_report_ara_rac', 'v4_motogp_report_ara_spr', 'v4_motogp_report_bra_rac', 'v4_motogp_report_bra_spr', 'v4_motogp_report_cat_rac', 'v4_motogp_report_cat_rac2', 'v4_motogp_report_cat_spr', 'v4_motogp_report_cze_rac', 'v4_motogp_report_cze_spr', 'v4_motogp_report_fra_rac', 'v4_motogp_report_fra_spr', 'v4_motogp_report_gbr_rac', 'v4_motogp_report_gbr_spr', 'v4_motogp_report_ger_rac', 'v4_motogp_report_ger_spr', 'v4_motogp_report_hun_rac', 'v4_motogp_report_hun_spr', 'v4_motogp_report_ita_rac', 'v4_motogp_report_ita_spr', 'v4_motogp_report_ned_rac', 'v4_motogp_report_ned_spr', 'v4_motogp_report_rsm_rac', 'v4_motogp_report_rsm_spr', 'v4_motogp_report_spa_rac', 'v4_motogp_report_spa_spr', 'v4_motogp_report_tha_rac', 'v4_motogp_report_tha_spr', 'v4_motogp_report_usa_rac', 'v4_motogp_report_usa_spr', 'v4_motogp_sessions_ara', 'v4_motogp_sessions_aus', 'v4_motogp_sessions_aut', 'v4_motogp_sessions_bra', 'v4_motogp_sessions_cat', 'v4_motogp_sessions_code', 'v4_motogp_sessions_cze', 'v4_motogp_sessions_fra', 'v4_motogp_sessions_gbr', 'v4_motogp_sessions_ger', 'v4_motogp_sessions_hun', 'v4_motogp_sessions_ina', 'v4_motogp_sessions_ita', 'v4_motogp_sessions_jpn', 'v4_motogp_sessions_mal', 'v4_motogp_sessions_ned', 'v4_motogp_sessions_por', 'v4_motogp_sessions_qat', 'v4_motogp_sessions_rsm', 'v4_motogp_sessions_spa', 'v4_motogp_sessions_tha', 'v4_motogp_sessions_usa', 'v4_motogp_sessions_val'])
display(table(references,'02_source_references'))

,source_id,url,raw_file,retrieved
0,v3_jolpica_races_2026,https://api.jolpi.ca/ergast/f1/2026/races/,data/raw/schedules/v3_jolpica_races_2026__11a7...,2026-09-14T12:16:09.727956+00:00
1,v4_laliga_2526,https://football-data.co.uk/mmz4281/2526/SP1.csv,data/raw/schedules/v4_laliga_2526__4ca2e285f3e...,2026-09-14T16:35:39.608774+00:00
2,v4_laliga_2627,https://football-data.co.uk/mmz4281/2627/SP1.csv,data/raw/schedules/v4_laliga_2627__deeed48a605...,2026-09-14T16:35:39.608820+00:00
3,v4_motogp_finished_events,https://api.motogp.pulselive.com/motogp/v1/res...,data/raw/schedules/v4_motogp_finished_events__...,2026-09-14T16:39:11.931768+00:00
4,v4_motogp_future_events,https://api.motogp.pulselive.com/motogp/v1/res...,data/raw/schedules/v4_motogp_future_events__59...,2026-09-14T16:39:11.931835+00:00
...,...,...,...,...
52,v4_motogp_sessions_rsm,https://api.motogp.pulselive.com/motogp/v1/res...,data/raw/schedules/v4_motogp_sessions_rsm__793...,2026-09-14T16:40:10.745707+00:00
53,v4_motogp_sessions_spa,https://api.motogp.pulselive.com/motogp/v1/res...,data/raw/schedules/v4_motogp_sessions_spa__a5c...,2026-09-14T16:40:09.217331+00:00
54,v4_motogp_sessions_tha,https://api.motogp.pulselive.com/motogp/v1/res...,data/raw/schedules/v4_motogp_sessions_tha__4a9...,2026-09-14T16:40:09.216771+00:00
55,v4_motogp_sessions_usa,https://api.motogp.pulselive.com/motogp/v1/res...,data/raw/schedules/v4_motogp_sessions_usa__52f...,2026-09-14T16:40:09.217275+00:00
